### OpenAI Agents SDK

This notebook shows how to use agents as tools.

Docs:
- https://openai.github.io/openai-agents-python/
- https://openai.github.io/openai-agents-python/tools/#agents-as-tools

In [ ]:
!pip install -q openai-agents python-dotenv

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from IPython.display import Markdown, display
from agents.tool import WebSearchTool
import os

##### Packages Used

- `openai-agents` for `Agent`, `Runner`, `trace`, and `WebSearchTool`
- `python-dotenv` for `load_dotenv`
- `IPython.display` for showing the final report

Load Credentials

In [ ]:
load_dotenv()

print("OpenAI API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Use either provider or account page to get started.
- https://aistudio.google.com/app/
- https://platform.openai.com/login

### Agentic Workflow

The `orchestrator` uses two agent tools to finish the task.

- `Researcher` gathers facts
- `Reporter` turns the research into a report

### Create Agent Tools

Define the two agents that will be used as tools.

In [ ]:
researcher_inst = "You are a skilled and resourceful researcher. Your job is to deeply investigate any assigned topic, intelligently leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. Your research should emphasize both recent developments and core facts, highlight significance and context, and clearly cite your sources when possible. Focus on accuracy, clarity, and actionable insight in your findings."

reporter_inst = "You are a meticulous analyst renowned for your keen attention to detail. You excel at transforming complex information into clear, concise, and actionable reports, making even the most intricate data accessible and understandable for your audience."

In [ ]:
researcher = Agent(
        name="Professional Researcher",
        instructions=researcher_inst,
        model="gpt-4o-mini",
        tools=[WebSearchTool()]
)

researcher_tool = researcher.as_tool("researcher", tool_description="Conducts in-depth research on a given topic and returns a list of facts in a bullet point format")



In [ ]:
reporter = Agent(
        name="Professional Reporter",
        instructions=reporter_inst,
        model="gpt-4.1-mini",
        tools=[WebSearchTool()]
)

reporter_tool = reporter.as_tool("reporter", tool_description="Analyzes the research findings from the research agent and generates a clear, concise, and actionable report that highlights key insights and recommendations.")

### Create Orchestration Agent

Create the main agent that coordinates both tools.

In [ ]:
orchestrator_agent = Agent(
        name="Orchestrator Agent",instructions="You are an orchestrator agent that coordinates the efforts of producing a comprehensive report on a given topic. You use the tools available to you to ensure that the final report is well-researched, insightful, and actionable, providing value to the end user.",
        tools=[researcher_tool, reporter_tool],
        model="gpt-4o-mini"
        )

#### Run the Demo

In [ ]:
topic = "FIFA World Cup 2026"

In [ ]:
with trace("create the report"):
    result = await Runner.run(orchestrator_agent, f"""
      Use the tools available to you to create a comprehensive report on the following topic: {topic}.
      
        1. Research the topic thoroughly and gather relevant information, ensuring to cite sources when possible.
        2. Use the findings to generate a clear, concise, and actionable report.
        
      The final output should be full comprehensive report. Only include the final report in your response.
        """)
    report_result = result.final_output

In [ ]:
display(Markdown(report_result))

#### Review the Traces
https://platform.openai.com/logs?api=traces